In [1]:
import os
import numpy as np
import pandas as pd
from to_numpy import *

def get_dataarray(config):
    match config['dataset_name']:
        case 'metr_la' | 'pems_bay':
            data_array, date_array = metrla_pemsbay(config['raw_data_path'])
        case 'pems04' | 'pems08':
            data_array, date_array = npz_file_pems0408(config['raw_data_path'], 
                                                       start_time=config['start_time'], 
                                                       time_windows=config['time_window'], 
                                                       lens=config['lens'])
        case 'taxibj13' | 'taxibj14' | 'taxibj15' | 'taxibj16':
            data_array, date_array = taxibj(config['raw_data_path'])
        case 'ettm1' | 'ettm2':
            data_array, date_array = ett(config['raw_data_path'])
        case 'exchange_rate':
            data_array, date_array = exchange_rate(config['raw_data_path'])
        case 'illness':
            data_array, date_array = illness(config['raw_data_path'])
        case 'traffic':
            data_array, date_array = traffic(config['raw_data_path'])
        case 'weather':
            data_array, date_array = weather(config['raw_data_path'])
        case 'electricity':
            data_array, date_array = electricity(config['raw_data_path'])
        case _:
            raise KeyError(f'dataset "{config["dataset_name"]}" not defined.')

    return data_array, date_array

In [2]:
def save_dataarray(config, data_array, date_array):
    # Unify data format
    if isinstance(data_array, (pd.DataFrame, pd.Series)):
        data_array = data_array.values
    else:
        data_array = np.asarray(data_array)
    
    # Unify date format
    if isinstance(date_array, (pd.DatetimeIndex, pd.Series)):
        date_array = date_array.values
    else:
        date_array = np.asarray(date_array)

    parsed_dates = pd.to_datetime(date_array, format='mixed', errors='coerce')
    date_array = parsed_dates.values.astype('datetime64[s]')

    # Validate data shapes
    assert data_array.shape[0] == date_array.shape[0], "Data and date length mismatch"
    assert data_array.shape[0] == config['lens'], f"Data length {data_array.shape[0]} does not match config length {config['lens']}"
    assert data_array.shape[1] == config['num_nodes'], f"Data node count {data_array.shape[1]} does not match config node count {config['num_nodes']}"
    assert data_array.shape[2] == config['num_channels'], f"Data channel count {data_array.shape[2]} does not match config channel count {config['num_channels']}"
    
    os.makedirs(config['bin_data_dir'], exist_ok=True)
    np.save(config['bin_data_path'], data_array)
    np.save(config['date_data_path'], date_array)

    new_data_array = np.load(config['bin_data_path'], allow_pickle=True)
    new_date_array = np.load(config['date_data_path'], allow_pickle=True)

    # Verify saved data consistency
    assert np.array_equal(data_array, new_data_array)
    assert np.array_equal(date_array, new_date_array)
    return new_data_array, new_date_array

In [ ]:
# Replace with actual path to storage
PATH_TO_STORAGE = "/path/to/storage"

In [7]:
# metr_la dataset
config = {
    'dataset_name': 'metr_la',
    'raw_data_path': PATH_TO_STORAGE + '/metr_la/metr_la.h5',
    'bin_data_dir': PATH_TO_STORAGE + '/metr_la/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/metr_la/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/metr_la/bin_data/date_array.npy',
    'start_time': '2012-03-01 00:00:00',
    'end_time': '2012-06-27 23:55:00',
    'time_window': 5,
    'lens': 34272,
    'num_nodes': 207,
    'num_channels': 1
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (34272, 207, 1)
Start time: 2012-03-01T00:00:00, End time: 2012-06-27T23:55:00


In [8]:
# pems_bay dataset
config = {
    'dataset_name': 'pems_bay',
    'raw_data_path': PATH_TO_STORAGE + '/pems_bay/pems_bay.h5',
    'bin_data_dir': PATH_TO_STORAGE + '/pems_bay/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/pems_bay/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/pems_bay/bin_data/date_array.npy',
    'start_time': '2017-01-01 00:00:00',
    'end_time': '2017-06-30 23:55:00',
    'time_window': 5,
    'lens': 52116,
    'num_nodes': 325,
    'num_channels': 1,
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (52116, 325, 1)
Start time: 2017-01-01T00:00:00, End time: 2017-06-30T23:55:00


In [9]:
# pems04 dataset
config = {
    'dataset_name': 'pems04',
    'raw_data_path': PATH_TO_STORAGE + '/pems04/pems04.npz',
    'bin_data_dir': PATH_TO_STORAGE + '/pems04/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/pems04/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/pems04/bin_data/date_array.npy',
    'start_time': '2017-01-01 00:00:00',
    'end_time': '2017-02-28 23:55:00',
    'time_window': 5,
    'lens': 16992,
    'num_nodes': 307,
    'num_channels': 3
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (16992, 307, 3)
Start time: 2017-01-01T00:00:00, End time: 2017-02-28T23:55:00


In [10]:
# pems08 dataset
config = {
    'dataset_name': 'pems08',
    'raw_data_path': PATH_TO_STORAGE + '/pems08/pems08.npz',
    'bin_data_dir': PATH_TO_STORAGE + '/pems08/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/pems08/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/pems08/bin_data/date_array.npy',
    'start_time': '2016-07-01 00:00:00',
    'end_time': '2016-08-31 23:55:00',
    'time_window': 5,
    'lens': 17856,
    'num_nodes': 170,
    'num_channels': 3

}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (17856, 170, 3)
Start time: 2016-07-01T00:00:00, End time: 2016-08-31T23:55:00


In [11]:
# taxibj13 dataset
config = {
    'dataset_name': 'taxibj13',
    'raw_data_path': PATH_TO_STORAGE + '/TaxiBJ/BJ13_M32x32_T30_InOut.h5',
    'bin_data_dir': PATH_TO_STORAGE + '/TaxiBJ/taxibj13/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj13/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj13/bin_data/date_array.npy',
    'start_time': '2013-07-01 00:00:00',
    'end_time': '2013-10-29 23:30:00',
    'time_window': 30,
    'lens': 4888,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (4888, 1024, 2)
Start time: 2013-07-01T00:00:00, End time: 2013-10-29T23:30:00


In [12]:
# taxibj14 dataset
config = {
    'dataset_name': 'taxibj14',
    'raw_data_path': PATH_TO_STORAGE + '/TaxiBJ/BJ14_M32x32_T30_InOut.h5',
    'bin_data_dir': PATH_TO_STORAGE + '/TaxiBJ/taxibj14/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj14/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj14/bin_data/date_array.npy',
    'start_time': '2014-03-01 00:00:00',
    'end_time': '2014-06-27 11:30:00',
    'time_window': 30,
    'lens': 4780,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (4780, 1024, 2)
Start time: 2014-03-01T00:00:00, End time: 2014-06-27T11:30:00


In [13]:
# taxibj15 dataset
config = {
    'dataset_name': 'taxibj15',
    'raw_data_path': PATH_TO_STORAGE + '/TaxiBJ/BJ15_M32x32_T30_InOut.h5',
    'bin_data_dir': PATH_TO_STORAGE + '/TaxiBJ/taxibj15/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj15/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj15/bin_data/date_array.npy',
    'start_time': '2015-03-01 00:00:00',
    'end_time': '2015-06-30 23:30:00',
    'time_window': 30,
    'lens': 5596,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (5596, 1024, 2)
Start time: 2015-03-01T00:00:00, End time: 2015-06-30T23:30:00


In [14]:
# taxibj16 dataset
config = {
    'dataset_name': 'taxibj16',
    'raw_data_path': PATH_TO_STORAGE + '/TaxiBJ/BJ16_M32x32_T30_InOut.h5',
    'bin_data_dir': PATH_TO_STORAGE + '/TaxiBJ/taxibj16/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj16/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/TaxiBJ/taxibj16/bin_data/date_array.npy',

    'time_window': 30,
    'lens': 7220,
    'num_nodes': 1024,
    'num_channels': 2
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (7220, 1024, 2)
Start time: 2015-11-01T00:00:00, End time: 2016-04-10T00:00:00


In [15]:
# ettm1 dataset
config = {
    'dataset_name': 'ettm1',
    'raw_data_path': PATH_TO_STORAGE + '/ETT-small/ETTm1.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/ETT-small/ettm1/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/ETT-small/ettm1/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/ETT-small/ettm1/bin_data/date_array.npy',
    'start_time': '2016-07-01 00:00:00',
    'end_time': '2018-06-26 19:45:00',
    'time_window': 15,
    'lens': 69680,
    'num_nodes': 1,
    'num_channels': 7
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (69680, 1, 7)
Start time: 2016-07-01T00:00:00, End time: 2018-06-26T19:45:00


In [16]:
# ettm2 dataset
config = {
    'dataset_name': 'ettm2',
    'raw_data_path': PATH_TO_STORAGE + '/ETT-small/ETTm2.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/ETT-small/ettm2/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/ETT-small/ettm2/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/ETT-small/ettm2/bin_data/date_array.npy',
    'start_time': '2016-07-01 00:00:00',
    'end_time': '2018-06-26 19:45:00',
    'time_window': 15,
    'lens': 69680,
    'num_nodes': 1,
    'num_channels': 7
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (69680, 1, 7)
Start time: 2016-07-01T00:00:00, End time: 2018-06-26T19:45:00


In [17]:
# exchange_rate dataset
config = {
    'dataset_name': 'exchange_rate',
    'raw_data_path': PATH_TO_STORAGE + '/exchange_rate/exchange_rate.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/exchange_rate/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/exchange_rate/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/exchange_rate/bin_data/date_array.npy',
    'start_time': '1990-01-01 00:00:00',
    'end_time': '2010-10-10 00:00:00',
    'time_window': 1440,
    'lens': 7588,
    'num_nodes': 1,
    'num_channels': 8
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (7588, 1, 8)
Start time: 1990-01-01T00:00:00, End time: 2010-10-10T00:00:00


In [18]:
# illness dataset
config = {
    'dataset_name': 'illness',
    'raw_data_path': PATH_TO_STORAGE + '/illness/national_illness.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/illness/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/illness/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/illness/bin_data/date_array.npy',
    'start_time': '2002-01-01 00:00:00',
    'end_time': '2020-06-30 00:00:00',
    'time_window': 10080,
    'lens': 966,
    'num_nodes': 1,
    'num_channels': 7
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (966, 1, 7)
Start time: 2002-01-01T00:00:00, End time: 2020-06-30T00:00:00


In [19]:
# traffic dataset
config = {
    'dataset_name': 'traffic',
    'raw_data_path': PATH_TO_STORAGE + '/traffic/traffic.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/traffic/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/traffic/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/traffic/bin_data/date_array.npy',
    'start_time': '2016-07-01 02:00:00',
    'end_time': '2018-07-02 01:00:00',
    'time_window': 60,
    'lens': 17544,
    'num_nodes': 862,
    'num_channels': 1
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (17544, 862, 1)
Start time: 2016-07-01T02:00:00, End time: 2018-07-02T01:00:00


In [20]:
# weather dataset
config = {
    'dataset_name': 'weather',
    'raw_data_path': PATH_TO_STORAGE + '/weather/weather.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/weather/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/weather/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/weather/bin_data/date_array.npy',
    'start_time': '2020-01-01 00:10:00',
    'end_time': '2021-01-01 00:00:00',
    'time_window': 10,
    'lens': 52696,
    'num_nodes': 1,
    'num_channels': 21
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (52696, 1, 21)
Start time: 2020-01-01T00:10:00, End time: 2021-01-01T00:00:00


In [21]:
# electricity dataset
config = {
    'dataset_name': 'electricity',
    'raw_data_path': PATH_TO_STORAGE + '/electricity/electricity.csv',
    'bin_data_dir': PATH_TO_STORAGE + '/electricity/bin_data',
    'bin_data_path': PATH_TO_STORAGE + '/electricity/bin_data/data_array.npy',
    'date_data_path': PATH_TO_STORAGE + '/electricity/bin_data/date_array.npy',
    'start_time': '2016-07-01 02:00:00',
    'end_time': '2019-07-02 01:00:00',
    'time_window': 60,
    'lens': 26304,
    'num_nodes': 321,
    'num_channels': 1
}

data_array, date_array = get_dataarray(config)
new_data_array, new_date_array = save_dataarray(config, data_array, date_array)
print(f"Data array shape: {new_data_array.shape}")
print(f"Start time: {new_date_array[0]}, End time: {new_date_array[-1]}")

Data array shape: (26304, 321, 1)
Start time: 2016-07-01T02:00:00, End time: 2019-07-02T01:00:00
